# 第14章 人体姿态估计

本章节学习目标：

- 理解本章核心算法的**数学原理**
- 掌握算法的**手写实现**方法
- 学会使用 OpenCV 对应函数进行**工程实践**
- 通过编程练习加深对算法的理解

> **📌 学习建议**：先阅读概念说明，再动手编写代码，最后完成练习


In [ ]:
# -*- coding: utf-8 -*-
# 中文路径兼容的图像读写函数
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


# 代码实现

我们首先编写DeepPose的代码。

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class DeepPose(nn.Module):
    def __init__(self, num_keypoints=17, pretrained=True, num_stages=3):
        super(DeepPose, self).__init__()
        self.num_keypoints = num_keypoints
        self.num_stages = num_stages

        # 加载预训练的ResNet模型用于特征提取
        self.backbone = models.resnet50(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-2])

        # 为每个级联阶段定义回归层
        self.regression_layers = nn.ModuleList([
            self._make_regression_layer() for _ in range(num_stages)
        ])

        # 每个级联阶段的最终全连接层，用于关节点预测
        self.fc_layers = nn.ModuleList([
            nn.Linear(2048, num_keypoints * 2) for _ in range(num_stages)
        ])

    def _make_regression_layer(self):
        # 定义回归层，由几个卷积层组成
        return nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 2048, kernel_size=1)
        )

    def forward(self, x):
        # 使用骨干网络提取特征
        features = self.backbone(x)

        # 初始化关节点预测为零
        keypoint_preds = torch.zeros(x.size(0), self.num_keypoints * 2).to(x.device)

        for i in range(self.num_stages):
            # 将关节点预测结果与特征图拼接
            keypoint_map = keypoint_preds.view(x.size(0), self.num_keypoints, 2, 1, 1)
            keypoint_map = keypoint_map.expand(-1, -1, -1, features.size(2), features.size(3))
            features_with_keypoints = torch.cat([features, keypoint_map.view(x.size(0), -1, features.size(2), features.size(3))], dim=1)

            # 通过回归层进行细化
            regression_output = self.regression_layers[i](features_with_keypoints)

            # 将回归输出展平并通过全连接层
            regression_output = regression_output.view(x.size(0), -1)
            keypoint_preds += self.fc_layers[i](regression_output)

        # 返回关节点的最终位置
        return keypoint_preds.view(x.size(0), self.num_keypoints, 2)

接着，我们在COCO上对其进行训练。我们先导入必要的仓库以及库函数。

In [ ]:
!git clone thttps://github.com/Naman-ntc/Pytorch-Human-Pose-Estimation.git
pip install -r requirements.txt

设置好模型设置和COCO数据集的路径，开始对模型进行训练。

In [ ]:
!python main.py -DataConfig conf/datasets/coco.defconf -ModelConfig conf/models/DeepPose.defconf 

<div style="display: inline-block; margin-top: 10px;">
        <span style="color:red">[图片占位 - 需本地生成]</span> width=1000>
        <div style="color:orange; 
        display: block;
        color: #999;
        padding: 2px;"></div>
    </div>

由于训练输出较长，这里我们只展示开始训练的阶段。最后，我们在测试集上对模型进行测试，可发现最终的PCK可以达到57.5。

<div style="display: inline-block; margin-top: 10px;">
        <span style="color:red">[图片占位 - 需本地生成]</span> width=1000>
        <div style="color:orange; 
        display: block;
        color: #999;
        padding: 2px;"></div>
    </div>

<!-- 目前的困难：
1. 姿态估计的精确度问题：由于人体姿态估计系统使用的是多种视觉信息，如果它们不能够准确地追踪到人体的每个部位，那么估计出来的结果也会不准确。
1. 尺度不变性问题：尽管现有的人体姿态估计算法可以处理不同尺度的图像，但它们仍然存在尺度不变性问题，即当姿态发生变化时，估计的结果可能会受到影响。
1. 光照变化问题：由于光照变化会对视觉信息产生影响，因此人体姿态估计系统可能无法准确地识别不同光照环境下的人体部位。
1. 复杂背景问题：复杂的背景可能会干扰人体姿态估计系统的性能，使得它无法准确地识别人体部位。 -->


---

## 📝 
练习：本章算法手写实现与扩展



**练习目标**：基于本章所学内容，完成以下实践任务。

**要求**：
1. 手写实现本章的核心算法（不直接调用 OpenCV/PyTorch 对应函数）
2. 使用本章学习的方法处理至少 2 张不同的测试图像
3. 对比手写实现与现成库函数的结果差异
4. 分析算法参数对结果的影响
5. 撰写 200 字以上的实验报告


**💡 小提示**：
- 除 `cv_imread` / `cv_imwrite` 外，不直接调用 OpenCV 高层函数
- 使用 NumPy 进行矩阵运算
- 注意边界处理和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
```python
# 本章练习代码框架
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

# ============================================
# TODO: 在此处手写实现本章核心算法
# ============================================

# 示例框架：
# 1. 数据准备
# img = cv_imread('test_image.jpg')
# gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 手写算法实现
# def algorithm_manual(input_image, **params):
#     # TODO: 实现算法核心逻辑
#     # 要求：除 OpenCV 读写函数外，其余代码手写
#     return output

# 3. 对比验证
# result_manual = algorithm_manual(gray)
# result_library = cv2.XXX(gray)  # 对应库函数
# diff = np.abs(result_manual.astype(float) - result_library.astype(float))
# print(f"最大差异: {diff.max()}")

# 4. 参数敏感性分析
# for param in [param1, param2, param3]:
#     result = algorithm_manual(gray, param=param)
#     # 可视化结果变化

# 5. 实验报告
print("请完成上述练习并撰写实验报告")
```



### 💻 代码要点解释

1. **图像读取与保存**：使用自定义的 `cv_imread` / `cv_imwrite` 函数，解决 Windows 中文路径下 OpenCV 读写图像失败的问题

2. **算法核心**：手写实现的核心在于**不依赖现成库函数**，而是直接操作像素和矩阵运算

3. **对比验证**：通过与 OpenCV 对应函数的结果进行数值对比，验证手写实现的正确性

4. **参数分析**：调整算法参数，观察输出变化，理解每个参数的物理含义

5. **扩展思考**：尝试将算法应用到自己的图像上，或改进算法（如增加加速技巧）

---

</details>

---
